In [3]:
# Imports
import tkinter as tk
from tkinter import ttk, messagebox

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import numpy as np

import serial
import serial.tools.list_ports

# Config
INPUT_BITS = 16
INPUT_MAX = (1 << INPUT_BITS) - 1

BASELINE_BITS = 14
BASELINE_MAX = (1 << BASELINE_BITS) - 1

# Colors
BACKGROUND_COLOR = "#1e1e1e"
TEXT_COLOR = "#d4d4d4"
ENTRY_BACKGROUND = "#2d2d2d"
BUTTON_COLOR = "#007acc"

# Window
window = tk.Tk()
window.title("DUNE EIA Simulator")
window.configure(bg=BACKGROUND_COLOR)
window.geometry("1250x650")

canvas_plot = None

# Ports
def get_available_ports():
    ports = serial.tools.list_ports.comports()
    return [port.device for port in ports]


def refresh_ports():
    ports = get_available_ports()
    com_combobox["values"] = ports

    if ports:
        com_combobox.current(0)
    else:
        com_combobox.set("")
        messagebox.showwarning(
            "UART",
            "No COM ports were found."
        )


# UART
def enter_data():
    try:
        input1_model = int(input1_entry.get())
        input2_model = int(input2_entry.get())
        input3_model = int(input3_entry.get())
        baseline = int(baseline_entry.get())

        inputs = [
            ("Input I", input1_model),
            ("Input II", input2_model),
            ("Input III", input3_model)
        ]

        for name, value in inputs:
            if not 0 <= value <= INPUT_MAX:
                messagebox.showerror(
                    "Error",
                    f"{name} must be between 0 and {INPUT_MAX}."
                )
                return

        if not 0 <= baseline <= BASELINE_MAX:
            messagebox.showerror(
                "Error",
                f"Baseline must be between 0 and {BASELINE_MAX}."
            )
            return

        port = com_combobox.get()

        if not port:
            messagebox.showerror(
                "Error",
                "Please select a COM port."
            )
            return

        input1_bytes = input1_model.to_bytes(2, byteorder="big", signed=False)
        input2_bytes = input2_model.to_bytes(2, byteorder="big", signed=False)
        input3_bytes = input3_model.to_bytes(2, byteorder="big", signed=False)
        baseline_bytes = baseline.to_bytes(2, byteorder="big", signed=False)

        frame_uart = (
            input1_bytes
            + input2_bytes
            + input3_bytes
            + baseline_bytes
        )

        print()
        print("==========================================")
        print("DATA TO SEND")
        print("==========================================")
        print(
            f"Input I   = {input1_model:5d}"
            f" = 0x{input1_model:04X}"
            f" = {input1_model:016b}"
        )
        print(
            f"Input II  = {input2_model:5d}"
            f" = 0x{input2_model:04X}"
            f" = {input2_model:016b}"
        )
        print(
            f"Input III = {input3_model:5d}"
            f" = 0x{input3_model:04X}"
            f" = {input3_model:016b}"
        )
        print(
            f"Baseline  = {baseline:5d}"
            f" = 0x{baseline:04X}"
            f" = {baseline:014b}"
        )

        print()
        print("UART FRAME:")
        print("Decimal:")
        print(list(frame_uart))

        print()
        print("Hexadecimal:")
        print(" ".join(f"{byte:02X}" for byte in frame_uart))

        print()
        print("Binary:")
        for i, byte in enumerate(frame_uart):
            print(f"Byte {i}: {byte:08b}")

        print("==========================================")

        with serial.Serial(
            port=port,
            baudrate=9600,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=1
        ) as ser:
            ser.reset_output_buffer()
            bytes_written = ser.write(frame_uart)
            ser.flush()

        messagebox.showinfo(
            "Success",
            "Data sent successfully.\n\n"
            f"Input I: {input1_model}\n"
            f"Input II: {input2_model}\n"
            f"Input III: {input3_model}\n"
            f"Baseline: {baseline}\n\n"
            f"Bytes sent: {bytes_written}"
        )

    except ValueError:
        messagebox.showerror(
            "Error",
            "All fields must contain valid integer values."
        )

    except serial.SerialException as e:
        messagebox.showerror(
            "UART Error",
            f"The serial port could not be used.\n\n{e}"
        )

    except Exception as e:
        messagebox.showerror(
            "Error",
            f"An error occurred:\n\n{e}"
        )


# Plot
def plot_function():
    global canvas_plot

    try:
        baseline = float(baseline_entry.get())
        input1_model = float(input1_entry.get())
        input2_model = float(input2_entry.get())
        input3_model = float(input3_entry.get())

    except ValueError:
        messagebox.showerror(
            "Error",
            "All fields must contain numeric values."
        )
        return

    samples = 1500

    t = np.zeros(samples)
    t[62] = input1_model
    t[262] = input2_model
    t[362] = input3_model

    d = [
        1,
        -3.937810904770624,
        6.059560578445686,
        -4.543872307687556,
        1.660365605946772,
        -0.238242823642825
    ]

    n = [
        0,
        -1.589921349755122,
        3.245061777284015,
        -1.961963726121936,
        0.548466823725762,
        -0.241643424903580
    ]

    y = np.zeros(samples)

    for i in range(6, samples):
        den = (
            y[i - 1] * d[1]
            + y[i - 2] * d[2]
            + y[i - 3] * d[3]
            + y[i - 4] * d[4]
            + y[i - 5] * d[5]
        )

        num = (
            t[i] * n[0]
            + t[i - 1] * n[1]
            + t[i - 2] * n[2]
            + t[i - 3] * n[3]
            + t[i - 4] * n[4]
            + t[i - 5] * n[5]
        )

        y[i] = (num - den) / d[0]

    y += baseline

    if canvas_plot is not None:
        canvas_plot.get_tk_widget().destroy()
        canvas_plot = None

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(y, linewidth=1.5)

    ax.set_xlabel(
        "Sample",
        fontname="Times New Roman",
        fontsize=14
    )
    ax.set_ylabel(
        "Amplitude",
        fontname="Times New Roman",
        fontsize=14
    )
    ax.set_title(
        "Generated Signal",
        fontname="Times New Roman",
        fontsize=14
    )

    ax.grid(True)
    ax.tick_params(axis="both", labelsize=12)

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname("Times New Roman")

    fig.tight_layout()

    canvas_plot = FigureCanvasTkAgg(fig, master=plot_frame)
    canvas_plot.draw()
    canvas_plot.get_tk_widget().pack(
        fill="both",
        expand=True,
        padx=10,
        pady=10
    )

    plt.close(fig)


# Style
style = ttk.Style()
style.theme_use("clam")

style.configure(
    "TButton",
    background=BUTTON_COLOR,
    foreground=TEXT_COLOR
)

style.configure(
    "TCombobox",
    fieldbackground=ENTRY_BACKGROUND,
    background=ENTRY_BACKGROUND,
    foreground=TEXT_COLOR
)

style.configure(
    "TLabel",
    background=BACKGROUND_COLOR,
    foreground=TEXT_COLOR
)

# Interface
frame = tk.Frame(
    window,
    bg=BACKGROUND_COLOR
)

frame.pack(
    fill="both",
    expand=True
)

uart_config = tk.LabelFrame(
    frame,
    text="UART Configuration",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

uart_config.grid(
    row=0,
    column=0,
    padx=20,
    pady=20,
    sticky="ew"
)

com_label_frame = tk.Label(
    uart_config,
    text="COM Port",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

com_label_frame.grid(
    row=0,
    column=0,
    padx=10,
    pady=5
)

com_combobox = ttk.Combobox(
    uart_config,
    state="readonly",
    style="TCombobox",
    width=15
)

com_combobox.grid(
    row=1,
    column=0,
    padx=10,
    pady=5
)

refresh_button = ttk.Button(
    uart_config,
    text="Refresh COM Ports",
    command=refresh_ports,
    style="TButton"
)

refresh_button.grid(
    row=1,
    column=1,
    padx=10,
    pady=5
)

parameters_frame = tk.LabelFrame(
    frame,
    text="Model Parameters",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

parameters_frame.grid(
    row=1,
    column=0,
    sticky="ew",
    padx=20,
    pady=10
)

input1_label = tk.Label(
    parameters_frame,
    text="Input I - 16 bits (0-65535)",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

input1_label.grid(row=0, column=0)

input1_entry = tk.Entry(
    parameters_frame,
    bg=ENTRY_BACKGROUND,
    fg=TEXT_COLOR,
    insertbackground=TEXT_COLOR
)

input1_entry.grid(row=1, column=0)

input2_label = tk.Label(
    parameters_frame,
    text="Input II - 16 bits (0-65535)",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

input2_label.grid(row=0, column=1)

input2_entry = tk.Entry(
    parameters_frame,
    bg=ENTRY_BACKGROUND,
    fg=TEXT_COLOR,
    insertbackground=TEXT_COLOR
)

input2_entry.grid(row=1, column=1)

input3_label = tk.Label(
    parameters_frame,
    text="Input III - 16 bits (0-65535)",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

input3_label.grid(row=0, column=2)

input3_entry = tk.Entry(
    parameters_frame,
    bg=ENTRY_BACKGROUND,
    fg=TEXT_COLOR,
    insertbackground=TEXT_COLOR
)

input3_entry.grid(row=1, column=2)

baseline_label = tk.Label(
    parameters_frame,
    text="Baseline - 14 bits (0-16383)",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

baseline_label.grid(row=0, column=3)

baseline_entry = tk.Entry(
    parameters_frame,
    bg=ENTRY_BACKGROUND,
    fg=TEXT_COLOR,
    insertbackground=TEXT_COLOR
)

baseline_entry.grid(row=1, column=3)

for widget in parameters_frame.winfo_children():
    widget.grid_configure(padx=10, pady=5)

input1_entry.insert(0, "1000")
input2_entry.insert(0, "1000")
input3_entry.insert(0, "1000")
baseline_entry.insert(0, "8200")

actions_frame = tk.LabelFrame(
    frame,
    text="Actions",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

actions_frame.grid(
    row=2,
    column=0,
    sticky="ew",
    padx=20,
    pady=10
)

send_button = ttk.Button(
    actions_frame,
    text="Send Data",
    command=enter_data,
    style="TButton"
)

send_button.grid(
    row=0,
    column=0,
    sticky="ew",
    padx=10,
    pady=5
)

plot_button = ttk.Button(
    actions_frame,
    text="Generate Plot",
    command=plot_function,
    style="TButton"
)

plot_button.grid(
    row=1,
    column=0,
    sticky="ew",
    padx=10,
    pady=5
)

actions_frame.grid_columnconfigure(0, weight=1)

plot_frame = tk.LabelFrame(
    frame,
    text="Generated Signal",
    bg=BACKGROUND_COLOR,
    fg=TEXT_COLOR
)

plot_frame.grid(
    row=0,
    column=1,
    rowspan=3,
    padx=20,
    pady=20,
    sticky="nsew"
)

frame.grid_columnconfigure(0, weight=1)
frame.grid_columnconfigure(1, weight=3)

frame.grid_rowconfigure(0, weight=1)
frame.grid_rowconfigure(1, weight=1)
frame.grid_rowconfigure(2, weight=1)

refresh_ports()
window.mainloop()
